# LASH — Colab Training Notebook

Workflow:
1. Clone repo from GitHub → always uses latest code
2. Set secrets (API keys) in Colab Secrets panel (🔑)
3. Collect data → Stage 1 SFT → Stage 2 GRPO → Eval
4. Push notebook with outputs back to GitHub as a results record

**Models** are saved to HuggingFace Hub (or Drive as fallback).

## 0. GPU check

In [ ]:
import subprocess

try:
    result = subprocess.run(
        ['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv,noheader'],
        capture_output=True, text=True
    )
    gpu_info = result.stdout.strip()
    print('GPU:', gpu_info)
    vram_mb = int(gpu_info.split(',')[1].strip().split()[0])
    LOAD_4BIT = vram_mb < 14000
    print(f'VRAM: {vram_mb} MB  ->  LOAD_4BIT={LOAD_4BIT}')
except FileNotFoundError:
    print('nvidia-smi not found — no GPU attached.')
    print('Go to Runtime > Change runtime type > T4 GPU and re-run.')
    LOAD_4BIT = False

_4bit_flag = '--load-in-4bit' if LOAD_4BIT else ''

## 1. Clone repo

In [ ]:
import os

REPO_URL = 'https://github.com/Azalea129/negotiation-env.git'
REPO_DIR = '/content/negotiation-env'

if os.path.exists(REPO_DIR):
    !git -C {REPO_DIR} pull
else:
    !git clone {REPO_URL} {REPO_DIR}

%cd {REPO_DIR}
!git log --oneline -5

## 2. Install dependencies

In [ ]:
!pip install -q -r requirements.txt
!pip install -q bitsandbytes

## 3. Configuration

Add the following keys in **Colab Secrets** (🔑 left sidebar) before running:
- `OPENAI_API_KEY`
- `HF_TOKEN`
- `GITHUB_TOKEN` (optional — for pushing results)

In [ ]:
import os
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
os.environ['HF_TOKEN']       = userdata.get('HF_TOKEN')
try:
    GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
except Exception:
    GITHUB_TOKEN = ''

# ── Models ───────────────────────────────────────────────────────────────────
TEACHER_MODEL  = 'gpt-4o-mini'
STUDENT_MODEL  = 'meta-llama/Meta-Llama-3-8B-Instruct'
HF_REPO        = ''   # e.g. 'your-username/lash-negotiation'  (leave empty to skip)

# ── Scale ────────────────────────────────────────────────────────────────────
N_EPISODES_A2A   = 200   # A2A negotiation episodes
N_EPISODES_MULTI = 50    # per game type

# ── Paths ────────────────────────────────────────────────────────────────────
DATA_DIR    = 'data'
STAGE1_DIR  = 'checkpoints/stage1'
STAGE2_DIR  = 'checkpoints/stage2'

print('Teacher:', TEACHER_MODEL)
print('Student:', STUDENT_MODEL)
print('4-bit  :', LOAD_4BIT)
print('GitHub token:', 'set' if GITHUB_TOKEN else 'not set')

## 4. (Optional) Local Llama teacher via vLLM

Skip if using OpenAI.

In [ ]:
# Uncomment to run local teacher
# !pip install -q vllm
# import subprocess, time, requests
# proc = subprocess.Popen(['python', '-m', 'vllm.entrypoints.openai.api_server',
#                          '--model', STUDENT_MODEL, '--port', '8000'])
# for _ in range(30):
#     try: requests.get('http://localhost:8000/health'); print('vLLM ready'); break
#     except: time.sleep(10)
# TEACHER_MODEL = STUDENT_MODEL
# os.environ['OPENAI_API_BASE'] = 'http://localhost:8000/v1'
print('Skipped — using remote OpenAI teacher')

## 5. Data collection

In [ ]:
# 5a. A2A negotiation
!python run_collection.py \
    --n {N_EPISODES_A2A} --model {TEACHER_MODEL} \
    --output-dir {DATA_DIR} --seed 42

In [ ]:
# 5b. Multi-game (stag_hunt held out for zero-shot eval)
!python run_concordia_collection.py \
    --n {N_EPISODES_MULTI} --model {TEACHER_MODEL} \
    --games prisoners_dilemma public_goods haggling \
    --output-dir {DATA_DIR} --seed 100

In [ ]:
# 5c. Stats
from lash.data_collector import collection_stats
s = collection_stats(DATA_DIR)
print(f"Episodes       : {s['episodes']}")
print(f"Deal rate      : {s['deal_rate']:.1%}")
print(f"Training pairs : {s['training_pairs']}")

## 6. Stage 1 — SFT

In [ ]:
!python train_stage1.py \
    --data-dir {DATA_DIR} --model {STUDENT_MODEL} \
    --output-dir {STAGE1_DIR} \
    --batch-size 2 --grad-accum 16 --epochs 3 \
    --oracle-decay-steps 2000 \
    {_4bit_flag}

## 7. Stage 2 — GRPO

In [ ]:
!python train_stage2.py \
    --stage1-dir {STAGE1_DIR}/final --model {STUDENT_MODEL} \
    --output-dir {STAGE2_DIR} \
    --n-iterations 300 --n-configs 4 --G 8 \
    --lambda-kl 0.01 --save-every 50 \
    {_4bit_flag}

## 8. Evaluation

In [ ]:
!python -m eval.zero_shot    --stage2-dir {STAGE2_DIR}/final --model {STUDENT_MODEL} {_4bit_flag}
!python -m eval.equilibrium  --stage2-dir {STAGE2_DIR}/final --model {STUDENT_MODEL} {_4bit_flag}
!python -m eval.ablation     --full-lash {STAGE2_DIR}/final --stage1-only {STAGE1_DIR}/final --model {STUDENT_MODEL} {_4bit_flag}

## 9. Save model to HuggingFace Hub

In [ ]:
if HF_REPO:
    from huggingface_hub import HfApi
    api = HfApi(token=os.environ['HF_TOKEN'])
    api.upload_folder(
        folder_path=f'{STAGE2_DIR}/final',
        repo_id=HF_REPO,
        repo_type='model',
    )
    print(f'Uploaded to https://huggingface.co/{HF_REPO}')
else:
    print('HF_REPO not set — skipping upload.')

## 10. Push results notebook to GitHub

In [ ]:
import datetime
timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M')

if GITHUB_TOKEN:
    repo_with_token = REPO_URL.replace('https://', f'https://{GITHUB_TOKEN}@')
    !git config user.email 'colab@lash'
    !git config user.name  'LASH Colab'
    !git remote set-url origin {repo_with_token}
    !mkdir -p results
    !cp /content/negotiation-env/colab_train.ipynb results/run_{timestamp}.ipynb
    !git add results/run_{timestamp}.ipynb
    !git commit -m "results: training run {timestamp}"
    !git push origin master
    print(f'Pushed results/run_{timestamp}.ipynb to GitHub')
else:
    print('GITHUB_TOKEN not set.')
    print('To save results: File > Download > Download .ipynb')